In [1]:
# compute old FR average and new FR average network and nFR value 
# param nedge
import os
import sys  
import pandas as pd
import hgt_gcn
import util
import copy


hgt_file = '../../hgt_abd_metadata/ERP010700.HGT.v2.short.csv'
metadata_file = '../../hgt_abd_metadata/ERP010700.metadata.v2.short10.tsv'
abd_file = '../../hgt_abd_metadata/ERP010700.merged.short10.tsv'
sp_file = '../../hgt_abd_metadata/genome_species.tsv'
db_dir = '../../HGT_demo_file/DB.genome_annotation'
gcn_file = '../../GCN_s.tsv'
d_file = '../../sp_d.tsv'
top_n = 100
groupid = 'phenotype'
method = 'wilcox.test'

hgt_df = pd.read_csv(hgt_file, index_col=None, header=0)
metadata_df = pd.read_csv(metadata_file, index_col=None, header=0, sep='\t')
abd_df = pd.read_csv(abd_file, index_col=0, header=0, sep='\t')
sp_df = pd.read_csv(sp_file, index_col=0, header=0, sep='\t')
gcn_df = pd.read_csv(gcn_file, index_col=0, header=0, sep='\t')
sp_d = pd.read_csv(d_file, index_col=0, header=0, sep='\t')

group = util.metadata2gf(metadata_file,groupid)
if not util.check_valid(group, abd_df):
    exit(2)


In [2]:
pheno_set = list(set(group[groupid]))
g1 = pheno_set[0]
g2 = pheno_set[1]
pheno_samples = {}
pheno_samples[g1] = list(group[group[groupid] == g1].index)
pheno_samples[g2] = list(group[group[groupid] == g2].index)


In [3]:
abd_df = hgt_gcn.multi_sample_normalize(abd_df)
genome_ko = hgt_gcn.ko_df(hgt_df, db_dir)
sp_ko_df = hgt_gcn.hgt2sp_ko(sp_df, genome_ko)


In [4]:
nfr_result_df = pd.DataFrame(columns=['sample', 'nFR', 'adj_nFR', 'group'])
avg_fr_dict = {}
avg_adj_fr_dict = {}
for g, slist in pheno_samples.items():
    # multi sample test
    sum_fr_net = pd.DataFrame()
    sum_adj_fr_net = pd.DataFrame()
    for sname in slist:
        nfr_result_df.loc[sname, 'sample'] = sname
        part_abd_df = abd_df[sname]
        part_abd_df = part_abd_df[part_abd_df > 0]
        tmp_abd = list(part_abd_df.index)
        part_df = sp_ko_df[sp_ko_df['sample'] == sname][['sp1', 'sp2', 'ko', 'num']]
        common_sp = list(set(tmp_abd).intersection(set(gcn_df.index)))
        tmp_d = sp_d.loc[common_sp, common_sp]
        # original fr
        nfr_value, fr_df, profile = hgt_gcn.nfr(tmp_d, abd_df, sname)
        nfr_result_df.loc[sname, 'nFR'] = nfr_value
        nfr_result_df.loc[sname, 'group'] = g
        # align and add to sum nfr net
        sum_fr_net = hgt_gcn.net_sum(sum_fr_net, fr_df)
        if len(part_df)>0:
            new_gcn_df, effect_list = hgt_gcn.hgt_adjust_gcn(gcn_df, part_df)
            effect_list = list(set(effect_list).intersection(set(common_sp)))
            tmp_gcn = gcn_df.T[common_sp]
            if len(effect_list) < 20:
                new_d = hgt_gcn.adjust_d(tmp_d, tmp_gcn, effect_list)
            else:
                new_d = hgt_gcn.make_d(new_gcn_df.loc[common_sp,])
        
            nfr_value, fr_df, profile = hgt_gcn.nfr(new_d, abd_df, sname)
            nfr_result_df.loc[sname, 'adj_nFR'] = nfr_value
            sum_adj_fr_net = hgt_gcn.net_sum(sum_adj_fr_net, fr_df)
        else:
            nfr_result_df.loc[sname, 'adj_nFR'] = nfr_value
            sum_adj_fr_net = hgt_gcn.net_sum(sum_adj_fr_net, fr_df)
    avg_fr_net = sum_fr_net / len(slist)
    avg_adj_fr_net = sum_adj_fr_net / len(slist)
    avg_fr_dict[g] = copy.deepcopy(avg_fr_net)
    avg_adj_fr_dict[g] = copy.deepcopy(avg_adj_fr_net)


In [19]:
cols = [
    'group1',
    'group2',
    'g1_mean',
    'g2_mean',
    'g1_variance',
    'g2_variance',
    'g1_occ',
    'g2_occ',
    'g1_n',
    'g2_n',
    'p_value',
    'g1/g2', 
    'enriched',
    'hgt_adjusted']


p_df = pd.DataFrame(columns=cols)
p_df.loc[0, 'group1'] = g1
p_df.loc[0, 'group2'] = g2
g1_v = nfr_result_df[nfr_result_df['group'] == g1]['nFR'].values.astype(float)
g2_v = nfr_result_df[nfr_result_df['group'] == g2]['nFR'].values.astype(float)
p_df.loc[0, 'g1_mean'] = g1_v.mean()
p_df.loc[0, 'g2_mean'] = g2_v.mean()
if p_df.loc[0, 'g1_mean'] > p_df.loc[0, 'g2_mean']:
    p_df.loc[0, 'enriched'] = g1
else:
    p_df.loc[0, 'enriched'] = g2
p_df.loc[0, 'g1/g2'] = p_df.loc[0, 'g1_mean']/p_df.loc[0, 'g2_mean']
p_df.loc[0, 'g1_variance'] = g1_v.var()
p_df.loc[0, 'g2_variance'] = g2_v.var()
p_df.loc[0, 'g1_occ'] = len(g1_v[g1_v > 0])/len(g1_v)
p_df.loc[0, 'g2_occ'] = len(g2_v[g2_v > 0])/len(g2_v)
p_df.loc[0, 'g1_n'] = len(g1_v)
p_df.loc[0, 'g2_n'] = len(g2_v)
p_df.loc[0, 'adjusted'] = 'False'
p_df.loc[0, 'p_value'] = hgt_gcn.test(method, g1_v, g2_v)


p_df.loc[1, 'group1'] = g1
p_df.loc[1, 'group2'] = g2
g1_v = nfr_result_df[nfr_result_df['group'] == g1]['adj_nFR'].values.astype(float)
g2_v = nfr_result_df[nfr_result_df['group'] == g2]['adj_nFR'].values.astype(float)
p_df.loc[1, 'g1_mean'] = g1_v.mean()
p_df.loc[1, 'g2_mean'] = g2_v.mean()
if p_df.loc[1, 'g1_mean'] > p_df.loc[1, 'g2_mean']:
    p_df.loc[1, 'enriched'] = g1
else:
    p_df.loc[1, 'enriched'] = g2
p_df.loc[1, 'g1/g2'] = p_df.loc[1, 'g1_mean']/p_df.loc[0, 'g2_mean']
p_df.loc[1, 'g1_variance'] = g1_v.var()
p_df.loc[1, 'g2_variance'] = g2_v.var()
p_df.loc[1, 'g1_occ'] = len(g1_v[g1_v > 0])/len(g1_v)
p_df.loc[1, 'g2_occ'] = len(g2_v[g2_v > 0])/len(g2_v)
p_df.loc[1, 'g1_n'] = len(g1_v)
p_df.loc[1, 'g2_n'] = len(g2_v)
p_df.loc[1, 'hgt_adjusted'] = 'True'
p_df.loc[1, 'p_value'] = hgt_gcn.test(method, g1_v, g2_v)




In [16]:
p_df

,group1,group2,g1_mean,g2_mean,g1_variance,g2_variance,g1_occ,g2_occ,g1_n,g2_n,p_value,g1/g2,enriched,adjusted
0,D006262,D001249,0.356357,0.320273,0.000186,0.000117,1.0,1.0,7,2,0.055556,1.112667,D006262,False
1,D006262,D001249,0.356357,0.320237,0.000186,0.000117,1.0,1.0,7,2,0.055556,1.112667,D006262,True


In [17]:
len(g1_v)

7

In [12]:
len(g2_v)

7

In [13]:
g1_v

array([0.35390641, 0.38241042, 0.35666335, 0.35990402, 0.3612419 ,
       0.34542886, 0.33494426])